# Suite2p — Batch runner (shared ops, multiple sessions)

Load a pre-defined ops file once, then run `run_s2p(ops, db)` for each session.

## `ops` vs `db`
| | `ops` | `db` |
|--|--|--|
| **Contains** | All pipeline parameters | Session-specific overrides |
| **Scope** | Shared across sessions | One per session |
| **How it works** | Base dict | Keys in `db` overwrite the matching keys in `ops` |

Any valid `ops` key can appear in `db` to override it for that session only  
(e.g. `"fs": 15.0` if one session was recorded at a different rate).

**Reference:** Stringer, Ki et al. 2026 · https://github.com/MouseLand/suite2p

In [ ]:
# %pip install suite2p

import suite2p
from suite2p import run_s2p, default_ops
import numpy as np
import json
import time
import traceback
from pathlib import Path
from datetime import datetime

print(f"suite2p {suite2p.__version__}")

## 1. Load the shared ops

In [ ]:
OPS_FILE = r"/path/to/suite2p_ops_used.json"   # or ops.npy from any planeX/ folder

ops_path = Path(OPS_FILE)
assert ops_path.exists(), f"Ops file not found: {ops_path}"

if ops_path.suffix == ".json":
    with open(ops_path) as f:
        ops_loaded = json.load(f)
elif ops_path.suffix == ".npy":
    ops_loaded = np.load(ops_path, allow_pickle=True).item()
else:
    raise ValueError(f"Unsupported format: {ops_path.suffix}")

# Merge with default_ops() so any keys added in newer suite2p versions are present.
# Keys in the loaded file take priority over defaults.
ops = default_ops()
ops.update(ops_loaded)

print(f"Loaded {len(ops_loaded)} parameters from: {ops_path.name}")
print(f"\nKey parameters:")
for k in ["fs", "nplanes", "nchannels", "tau", "do_registration",
          "nonrigid", "sparse_mode", "threshold_scaling", "neucoeff"]:
    print(f"  {k:25s} = {ops.get(k)}")

## 2. Define the databases (one `db` per session)

Each `db` must contain at minimum `data_path` and `save_path0`.  
Any other key overrides the shared `ops` for that session only.

In [ ]:
dbs = [
    {
        "data_path":  [
            "/data/raw/mouse01/2026-01-15/run1",
            "/data/raw/mouse01/2026-01-15/run2",
            "/data/raw/mouse01/2026-01-15/run3",
        ],
        "save_path0": "/data/processed/mouse01/2026-01-15",
        "fast_disk":  "/scratch",
    },
    {
        "data_path":  [
            "/data/raw/mouse01/2026-01-16/run1",
            "/data/raw/mouse01/2026-01-16/run2",
        ],
        "save_path0": "/data/processed/mouse01/2026-01-16",
        "fast_disk":  "/scratch",
    },
    {
        "data_path":  [
            "/data/raw/mouse02/2026-01-20/run1",
            "/data/raw/mouse02/2026-01-20/run2",
        ],
        "save_path0": "/data/processed/mouse02/2026-01-20",
        "fs": 15.0,   # ← per-session override: this mouse was recorded at 15 Hz
    },
    # add more dbs here …
]

## 3. Run the pipeline

In [ ]:
results = []

for i, db in enumerate(dbs):
    label = db["data_path"][0]   # use the data path as a human-readable label
    print(f"\n{'─'*60}")
    print(f"[{i+1}/{len(dbs)}]  {label}")

    t0 = time.time()
    try:
        output_ops = run_s2p(ops=ops, db=db)
        elapsed = time.time() - t0
        results.append({"db": db, "status": "OK", "elapsed": elapsed,
                        "save": output_ops.get("save_path")})
        print(f"✓  {elapsed/60:.1f} min  →  {output_ops.get('save_path')}")

    except Exception as exc:
        elapsed = time.time() - t0
        results.append({"db": db, "status": "FAILED", "elapsed": elapsed,
                        "error": str(exc)})
        print(f"❌  FAILED after {elapsed/60:.1f} min")
        traceback.print_exc()   # print and continue — don't abort the batch

print(f"\n{'═'*60}")
ok     = [r for r in results if r["status"] == "OK"]
failed = [r for r in results if r["status"] != "OK"]
print(f"Done: {len(ok)}/{len(results)} succeeded, {len(failed)} failed")

## 4. Summary & QC

In [ ]:
import matplotlib.pyplot as plt

summary = []
for r in ok:
    for plane_dir in sorted(Path(r["save"]).parent.glob("plane*")):
        iscell_path = plane_dir / "iscell.npy"
        if not iscell_path.exists():
            continue
        iscell = np.load(iscell_path, allow_pickle=True)
        summary.append({
            "label":   Path(r["db"]["data_path"][0]).name + f" / {plane_dir.name}",
            "n_cell":  int(iscell[:, 0].sum()),
            "n_total": len(iscell),
        })

if summary:
    fig, ax = plt.subplots(figsize=(max(5, len(summary) * 1.2), 4))
    x = range(len(summary))
    n_cell    = [s["n_cell"]  for s in summary]
    n_noncell = [s["n_total"] - s["n_cell"] for s in summary]
    ax.bar(x, n_cell,    label="cells",     color="steelblue")
    ax.bar(x, n_noncell, bottom=n_cell, label="non-cells", color="lightgray")
    ax.set_xticks(list(x))
    ax.set_xticklabels([s["label"] for s in summary], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("ROI count")
    ax.set_title("Detected cells per session")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"\n{'Session':<45} {'Cells':>7} {'Total':>7} {'% cells':>8}")
    print("-" * 72)
    for s in summary:
        pct = 100 * s["n_cell"] / max(s["n_total"], 1)
        print(f"{s['label']:<45} {s['n_cell']:>7} {s['n_total']:>7} {pct:>7.1f}%")

In [ ]:
# Save run log
log = {
    "run_date":    datetime.now().isoformat(),
    "suite2p_ver": suite2p.__version__,
    "ops_file":    str(OPS_FILE),
    "results":     [{"data_path": r["db"]["data_path"],
                     "save_path": r["db"]["save_path0"],
                     "status":    r["status"],
                     "elapsed_min": round(r["elapsed"]/60, 2),
                     **({"error": r["error"]} if "error" in r else {})
                     } for r in results],
}
log_path = Path(dbs[0]["save_path0"]).parent / "batch_run_log.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2, default=str)
print(f"Run log → {log_path}")

---
## Tips

**Skip already-processed sessions:**
```python
dbs = [db for db in dbs
       if not (Path(db["save_path0"]) / "suite2p" / "plane0" / "iscell.npy").exists()]
```

**Retry only failed sessions:**
```python
failed_paths = {r["db"]["data_path"][0] for r in results if r["status"] != "OK"}
dbs = [db for db in dbs if db["data_path"][0] in failed_paths]
```

**Rerun detection only (skip registration):**
```python
db = {
    "data_path":  ["/data/processed/mouse01/2026-01-15/suite2p/plane0"],  # already-registered binary
    "save_path0": "/data/processed/mouse01/2026-01-15",
    "do_registration": 0,
}
```

**Common per-session overrides inside `db`:**

| Scenario | Key |
|---|---|
| Different frame rate | `"fs": 15.0` |
| Different indicator | `"tau": 0.3` |
| Multi-plane | `"nplanes": 4` |
| Two-channel | `"nchannels": 2` |
| Noisier session | `"threshold_scaling": 2.0` |
| Align to red channel | `"align_by_chan": 2` |